# 02 - Baseline Encoding Model (ResNet / VGGFace2)

This notebook loads Algonauts video stimuli, extracts final-layer VGGFace2 / InceptionResnetV1
features, and trains ridge-regression encoding models to predict fMRI responses per ROI.

The video folder contains 1102 files, but only the first 1000 correspond to training videos
with available fMRI labels. This notebook trims the video list to match the fMRI rows exactly
before fitting any encoding models.

#  Install Dependencies

In [1]:
!pip uninstall -y pillow >/dev/null 2>&1
!pip install "Pillow==10.2.0" "facenet-pytorch==2.6.0" nilearn decord python-dotenv --quiet

!pip install nilearn decord python-dotenv --quiet
!pip install facenet-pytorch --quiet


## Runtime note

After running the install cell above, restart the Colab runtime once:

**Runtime -> Restart runtime**

Then continue from the next cell downward.

# Repository setup

In [2]:
!git clone https://github.com/sossyh/ffa-dnn-ablation.git
%cd ffa-dnn-ablation

fatal: destination path 'ffa-dnn-ablation' already exists and is not an empty directory.
/content/ffa-dnn-ablation


## Load dataset link from Colab Secrets

Requires a secret named `DROPBOX_DATASET_LINK` set via the key icon in
the left sidebar, with notebook access turned on.

In [3]:
import os
from google.colab import userdata

os.environ["DROPBOX_DATASET_LINK"] = userdata.get("DROPBOX_DATASET_LINK")
print("Link loaded:", os.environ["DROPBOX_DATASET_LINK"] is not None)

Link loaded: True


# Import InceptionResBetV1,  ResNetVGGFace2Wrapper

In [4]:
import os
import sys
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm

sys.path.append(".")

from src.data_loading import (
    download_algonauts_data,
    load_all_rois,
    get_video_list,
    load_video_frames,
)
from src.models.resnet_vggface2_wrapper import ResNetVGGFace2Wrapper
from src.encoding_model import train_and_evaluate_encoding_model, region_mean_accuracy

from facenet_pytorch import InceptionResnetV1
print("Imports OK")

[transformers] Disabling PyTorch because PyTorch >= 2.4 is required but found 2.2.2
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


Imports OK


# Downloading Dataset

In [5]:
data_dir = "data/raw"
download_algonauts_data(data_dir=data_dir)
print("Dataset download step complete")

Download complete.
Dataset download step complete


# Verify Dataset Path

In [6]:
video_dir = os.path.join(data_dir, "AlgonautsVideos268_All_30fpsmax")
print("CWD:", os.getcwd())
print("video_dir:", video_dir)
print("exists:", os.path.isdir(video_dir))

CWD: /content/ffa-dnn-ablation
video_dir: data/raw/AlgonautsVideos268_All_30fpsmax
exists: True


# Load fMRI ROI Data

In [7]:
fmri_dir = "data/raw/participants_data_v2021/mini_track"
subject = "sub01"

roi_data = load_all_rois(fmri_dir, subject)

for roi_name, data in roi_data.items():
    print(f"{roi_name}: {data.shape}")

NUM_TRAIN_VIDEOS = roi_data["FFA"].shape[0]
print("Number of training videos:", NUM_TRAIN_VIDEOS)

V1: (1000, 232)
V2: (1000, 231)
V3: (1000, 261)
V4: (1000, 107)
LOC: (1000, 1843)
EBA: (1000, 351)
FFA: (1000, 68)
STS: (1000, 341)
PPA: (1000, 425)
Number of training videos: 1000


# Load and Trim Video List

In [8]:
video_paths = get_video_list(video_dir)
print("Total video files found:", len(video_paths))

video_paths = video_paths[:NUM_TRAIN_VIDEOS]

assert len(video_paths) == NUM_TRAIN_VIDEOS, (
    f"Video count ({len(video_paths)}) does not match fMRI row count ({NUM_TRAIN_VIDEOS})"
)

print("Using training videos:", len(video_paths))
print("First 3:", video_paths[:3])
print("Last 3:", video_paths[-3:])

Total video files found: 1102
Using training videos: 1000
First 3: ['data/raw/AlgonautsVideos268_All_30fpsmax/0001_0-0-1-6-7-2-8-0-17500167280.mp4', 'data/raw/AlgonautsVideos268_All_30fpsmax/0002_0-0-4-3146384004.mp4', 'data/raw/AlgonautsVideos268_All_30fpsmax/0003_0-0-8-1-2-4-0-0-3500812400.mp4']
Last 3: ['data/raw/AlgonautsVideos268_All_30fpsmax/0998_wc-8lYUtCY5Er7d_31.mp4', 'data/raw/AlgonautsVideos268_All_30fpsmax/0999_wc-QqJnzWt2uQtA_34.mp4', 'data/raw/AlgonautsVideos268_All_30fpsmax/1000_yt_R-1887PalKvzM_35.mp4']


# Wrapper Smoke Test:

Runing a quick one-video test before extracting features for the full dataset.

In [9]:
frames = load_video_frames(video_paths[0], num_frames=8)

resnet = ResNetVGGFace2Wrapper()
sample_acts = resnet.extract(frames)

for k, v in sample_acts.items():
    print(k, v.shape)

  0%|          | 0.00/107M [00:00<?, ?B/s]

ResNetVGGFace2Wrapper: hooked layers ['repeat_1', 'repeat_2', 'repeat_3', 'block8', 'avgpool_1a']
repeat_1 (73984,)
repeat_2 (57344,)
repeat_3 (16128,)
block8 (16128,)
avgpool_1a (1792,)


# Extract or Load Cached ResNet Features


In [10]:
os.makedirs("data/processed", exist_ok=True)

resnet_cache_path = "data/processed/resnet50_final_features.npy"
need_to_extract_resnet = True

if os.path.exists(resnet_cache_path):
    print("Found cached ResNet50 features, checking they match current video count...")
    cached_resnet = np.load(resnet_cache_path)
    cached_num_videos_resnet = cached_resnet.shape[0]

    if cached_num_videos_resnet == len(video_paths):
        print(f"Cache matches ({cached_num_videos_resnet} videos). Using cached ResNet50 features.")
        need_to_extract_resnet = False
        X_resnet = cached_resnet
    else:
        print(
            f"ResNet cache has {cached_num_videos_resnet} videos but current list has "
            f"{len(video_paths)}. Cache is stale, re-extracting."
        )

if need_to_extract_resnet:
    print("Extracting ResNet50 features (this will take a while)...")

    device = "cuda" if torch.cuda.is_available() else "cpu"
    resnet_model = ResNetVGGFace2Wrapper(device=device)

    resnet_features = []

    for video_path in tqdm(video_paths):
        frames = load_video_frames(video_path, num_frames=8)
        acts = resnet_model.extract(frames)

        final_layer = resnet_model.layers[-1]
        feats = acts[final_layer]

        resnet_features.append(feats)

    X_resnet = np.stack(resnet_features, axis=0)
    np.save(resnet_cache_path, X_resnet)

    print("Saved ResNet50 features to", resnet_cache_path)

print("ResNet50 feature shape:", X_resnet.shape)

Extracting ResNet50 features (this will take a while)...
ResNetVGGFace2Wrapper: hooked layers ['repeat_1', 'repeat_2', 'repeat_3', 'block8', 'avgpool_1a']


100%|██████████| 1000/1000 [11:55<00:00,  1.40it/s]

Saved ResNet50 features to data/processed/resnet50_final_features.npy
ResNet50 feature shape: (1000, 1792)


# Sanity check before running any regression



In [11]:
assert X_resnet.shape[0] == NUM_TRAIN_VIDEOS, (
    f"ResNet features have {X_resnet.shape[0]} rows but fMRI data has {NUM_TRAIN_VIDEOS} rows."
)

for roi_name, data in roi_data.items():
    assert data.shape[0] == NUM_TRAIN_VIDEOS, (
        f"ROI {roi_name} has {data.shape[0]} rows, expected {NUM_TRAIN_VIDEOS}."
    )

print("All shapes match. Safe to proceed.")

All shapes match. Safe to proceed.



# One-Region Test :FFA



In [12]:
Y_ffa = roi_data["FFA"]
voxel_scores_resnet = train_and_evaluate_encoding_model(X_resnet, Y_ffa)
print("Mean FFA accuracy:", region_mean_accuracy(voxel_scores_resnet))

Mean FFA accuracy: 0.17365899823307396


# Alpha-Tuned FFA Test

In [13]:
alphas = [1, 10, 100, 1000, 10000]

voxel_scores_resnet = train_and_evaluate_encoding_model(
    X_resnet, Y_ffa, alphas=alphas, n_folds=5, seed=42
)
mean_acc_resnet = region_mean_accuracy(voxel_scores_resnet)
print(f"ResNet final layer -> FFA (best alpha from {alphas}): {mean_acc_resnet:.4f}")

ResNet final layer -> FFA (best alpha from [1, 10, 100, 1000, 10000]): 0.1717


# ALL ROI Results

In [14]:
 alphas = [1, 10, 100, 1000, 10000]
results_resnet = []

for region_name, Y in roi_data.items():
    voxel_scores_resnet = train_and_evaluate_encoding_model(
        X_resnet, Y, alphas=alphas, n_folds=5, seed=42
    )
    mean_acc = region_mean_accuracy(voxel_scores_resnet)
    results_resnet.append({
        "layer": "vggface2_final",
        "region": region_name,
        "accuracy": mean_acc,
    })
    print(f"ResNet final -> {region_name}: {mean_acc:.4f}")

ResNet final -> V1: 0.0979
ResNet final -> V2: 0.1010
ResNet final -> V3: 0.0967
ResNet final -> V4: 0.0902
ResNet final -> LOC: 0.1485
ResNet final -> EBA: 0.1453
ResNet final -> FFA: 0.1717
ResNet final -> STS: 0.0846
ResNet final -> PPA: 0.1093


# `Saving the CSV File`

In [15]:
os.makedirs("results/tables/resnet50", exist_ok=True)

results_resnet_df = pd.DataFrame(results_resnet)
csv_path = "results/tables/resnet50/final_layer_accuracy_vggface2.csv"
results_resnet_df.to_csv(csv_path, index=False)

print("Saved ResNet/VGGFace2 ROI accuracies to:", csv_path)
print(results_resnet_df)

Saved ResNet/VGGFace2 ROI accuracies to: results/tables/resnet50/final_layer_accuracy_vggface2.csv
            layer region  accuracy
0  vggface2_final     V1  0.097912
1  vggface2_final     V2  0.100952
2  vggface2_final     V3  0.096672
3  vggface2_final     V4  0.090243
4  vggface2_final    LOC  0.148540
5  vggface2_final    EBA  0.145326
6  vggface2_final    FFA  0.171670
7  vggface2_final    STS  0.084568
8  vggface2_final    PPA  0.109300


In [16]:
rows = [
    "| ROI | Accuracy (VGGFace2 final layer) |",
    "|-----|---------------------------------|",
]

for r in results_resnet:
    rows.append(f"| {r['region']} | {r['accuracy']:.4f} |")

markdown_table = "\n".join(rows)
print(markdown_table)

| ROI | Accuracy (VGGFace2 final layer) |
|-----|---------------------------------|
| V1 | 0.0979 |
| V2 | 0.1010 |
| V3 | 0.0967 |
| V4 | 0.0902 |
| LOC | 0.1485 |
| EBA | 0.1453 |
| FFA | 0.1717 |
| STS | 0.0846 |
| PPA | 0.1093 |


# ## Notes

- This notebook is ResNet / VGGFace2 only.
- AlexNet extraction, AlexNet caches, and AlexNet result tables are intentionally excluded.
- The final exported table is saved at:

`results/tables/resnet50/final_layer_accuracy_vggface2.csv`